**Cell 1: Import Libraries and Load Renewal Calls Data**  
This cell imports the required libraries, reads the merged feature dataset, and selects only the renewal calls-related columns. It ensures the data is ready for renewal-call-specific churn analysis.  
- **Purpose**: Prepare the notebook environment and load the renewal call dataset.  
- **Key Libraries**: `pandas`, `numpy`, `scipy.stats`, `statsmodels.stats.multitest`.  
- **No hypothesis testing here** — this is data preparation.  
- **Expected output**: table preview of the selected renewal calls features.

In [9]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, mannwhitneyu, chi2_contingency
from statsmodels.stats.multitest import multipletests

df = pd.read_csv("../../data/04_merged/final_merged_features.csv")

# Select only renewal calls related columns
renewal_cols = [
    "co_ref", "total_calls", "last_call_date", "cutoff_date", "days_since_last_call",
    "calls_last_7", "calls_last_14", "calls_last_30", "serious_complaints", 
    "switch_intent", "cancel_intent", "price_discussions", "discount_requests", 
    "recent_call_ratio", "prospect_outcome"
]

df = df[renewal_cols].dropna(subset=["prospect_outcome"])

df.head()

,co_ref,total_calls,last_call_date,cutoff_date,days_since_last_call,calls_last_7,calls_last_14,calls_last_30,serious_complaints,switch_intent,cancel_intent,price_discussions,discount_requests,recent_call_ratio,prospect_outcome
0,AA0794,NaN,NaN,2025-06-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Won
1,AA0794,NaN,NaN,2025-06-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Won
2,AA0794,NaN,NaN,2025-06-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Won
3,AA0794,NaN,NaN,2024-06-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Won
4,AA0794,NaN,NaN,2024-06-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Won


**Cell 2: Create Churn Target**  
This cell converts the prospect outcome into a binary churn flag where 1 means "Churned" and 0 means retained.  
- **Purpose**: Create the dependent variable for statistical testing.  
- **No hypothesis test here** — this is target definition.  
- **Expected output**: counts for churned vs. non-churned customers.

In [10]:
df["target"] = (df["prospect_outcome"] == "Churned").astype(int)

df["target"].value_counts()

target
0    102310
1     15011
Name: count, dtype: int64

**Cell 3: Define Renewal Calls Features**  
This cell lists the numerical features that will be tested for churn association. It also defines any categorical features, although this notebook has none.  
- **Purpose**: Select the feature set for hypothesis testing.  
- **No hypothesis test here** — this is feature selection.  
- **Expected output**: no output, just variable definitions.

In [14]:
num_cols = [
    "total_calls",
    "days_since_last_call",
    "calls_last_7",
    "calls_last_14",
    "calls_last_30",
    "serious_complaints",
    "switch_intent",
    "cancel_intent",
    "price_discussions",
    "discount_requests",
    "recent_call_ratio"
]

cat_cols = []

**Cell 4: Hypothesis Testing for Numerical Renewal Call Features**  
This cell performs the actual statistical tests for each numerical renewal call feature.  
- **Null Hypothesis (H₀)**: There is no difference in the feature distribution between churn and non-churn customers.  
- **Alternative Hypothesis (H₁)**: There is a difference in the feature distribution between churn and non-churn customers.  
- **Tests used**:  
  - t-test for mean differences  
  - Mann-Whitney U for distribution differences  
  - Cohen's d for effect size  
- **Expected output**: a DataFrame with mean values, p-values, and effect sizes for each numeric feature.

In [15]:
num_results = []

for col in num_cols:
    churn = df[df["target"] == 1][col].dropna()
    non_churn = df[df["target"] == 0][col].dropna()

    if len(churn) > 1 and len(non_churn) > 1:
        # Welch T-test
        t_stat, t_p = ttest_ind(churn, non_churn, equal_var=False)

        # Mann Whitney U
        u_stat, u_p = mannwhitneyu(churn, non_churn, alternative="two-sided")

        # Cohen's d
        pooled_std = np.sqrt((churn.var() + non_churn.var()) / 2)
        effect_size = (
            (churn.mean() - non_churn.mean()) / pooled_std
            if pooled_std != 0 else 0
        )

        num_results.append({
            "feature": col,
            "churn_mean": churn.mean(),
            "non_churn_mean": non_churn.mean(),
            "ttest_p_value": t_p,
            "mannwhitney_p_value": u_p,
            "effect_size": effect_size
        })

num_results = pd.DataFrame(num_results)
num_results

,feature,churn_mean,non_churn_mean,ttest_p_value,mannwhitney_p_value,effect_size
0,total_calls,6.270340,5.225754,2.732130e-51,2.302238e-82,0.163879
1,days_since_last_call,177.243767,281.575770,0.000000e+00,0.000000e+00,-0.560339
2,calls_last_7,0.140966,0.070178,2.640589e-36,1.479842e-78,0.152253
3,calls_last_14,0.318457,0.196694,1.283436e-37,3.674427e-58,0.151193
4,calls_last_30,0.620191,0.353494,2.309537e-58,1.003550e-85,0.193958
5,serious_complaints,0.033036,0.015051,8.461224e-08,8.931149e-21,0.067558
6,switch_intent,0.001334,0.001127,6.046253e-01,5.660411e-02,0.005161
7,cancel_intent,0.766903,0.180468,0.000000e+00,0.000000e+00,0.567343
8,price_discussions,0.304094,0.262497,2.809275e-06,2.679680e-16,0.051053
9,discount_requests,0.122089,0.090476,3.598342e-08,4.612966e-22,0.062271


**Cell 5: Collect Numerical Test Results**  
This cell builds the results DataFrame from the numerical feature tests, storing the computed p-values and effect sizes.  
- **Purpose**: Consolidate test outputs into a structured summary.  
- **Expected output**: DataFrame showing feature-wise t-test, Mann-Whitney, and effect size values.

In [16]:
cat_results = []

for col in cat_cols:
    cont_table = pd.crosstab(df[col], df["target"])

    if cont_table.shape[0] > 1 and cont_table.shape[1] > 1:
        chi2, p, dof, expected = chi2_contingency(cont_table)

        cat_results.append({
            "feature": col,
            "chi2_p_value": p
        })

cat_results = pd.DataFrame(cat_results)
cat_results

""


**Cell 6: Hypothesis Testing for Categorical Features**  
This cell tests whether any categorical renewal call variables are associated with churn.  
- **Null Hypothesis (H₀)**: The categorical feature is independent of churn.  
- **Alternative Hypothesis (H₁)**: The categorical feature is associated with churn.  
- **Test used**: Chi-square test of independence.  
- **Expected output**: a DataFrame with p-values for each categorical feature.  
- **Note**: If there are no categorical features, the result will be empty.

In [17]:
num_results["adjusted_p"] = multipletests(
    num_results["mannwhitney_p_value"],
    method="fdr_bh"
)[1]

num_results["significant"] = num_results["adjusted_p"] < 0.05

num_results.sort_values("adjusted_p")

,feature,churn_mean,non_churn_mean,ttest_p_value,mannwhitney_p_value,effect_size,adjusted_p,significant
1,days_since_last_call,177.243767,281.575770,0.000000e+00,0.000000e+00,-0.560339,0.000000e+00,True
7,cancel_intent,0.766903,0.180468,0.000000e+00,0.000000e+00,0.567343,0.000000e+00,True
4,calls_last_30,0.620191,0.353494,2.309537e-58,1.003550e-85,0.193958,3.679684e-85,True
0,total_calls,6.270340,5.225754,2.732130e-51,2.302238e-82,0.163879,6.331154e-82,True
2,calls_last_7,0.140966,0.070178,2.640589e-36,1.479842e-78,0.152253,3.255652e-78,True
10,recent_call_ratio,0.019335,0.010059,1.021882e-31,3.841942e-77,0.141111,7.043560e-77,True
3,calls_last_14,0.318457,0.196694,1.283436e-37,3.674427e-58,0.151193,5.774099e-58,True
9,discount_requests,0.122089,0.090476,3.598342e-08,4.612966e-22,0.062271,6.342828e-22,True
5,serious_complaints,0.033036,0.015051,8.461224e-08,8.931149e-21,0.067558,1.091585e-20,True
8,price_discussions,0.304094,0.262497,2.809275e-06,2.679680e-16,0.051053,2.947648e-16,True


**Cell 7: Correct for Multiple Testing**  
This cell adjusts the numerical test p-values using the Benjamini-Hochberg false discovery rate correction.  
- **Purpose**: Reduce false positives when multiple features are tested.  
- **Key output**: `adjusted_p` and `significant` columns.  
- **Decision rule**: A feature is significant if `adjusted_p < 0.05`.

In [18]:
significant_features = num_results[num_results["significant"] == True]
significant_features

,feature,churn_mean,non_churn_mean,ttest_p_value,mannwhitney_p_value,effect_size,adjusted_p,significant
0,total_calls,6.270340,5.225754,2.732130e-51,2.302238e-82,0.163879,6.331154e-82,True
1,days_since_last_call,177.243767,281.575770,0.000000e+00,0.000000e+00,-0.560339,0.000000e+00,True
2,calls_last_7,0.140966,0.070178,2.640589e-36,1.479842e-78,0.152253,3.255652e-78,True
3,calls_last_14,0.318457,0.196694,1.283436e-37,3.674427e-58,0.151193,5.774099e-58,True
4,calls_last_30,0.620191,0.353494,2.309537e-58,1.003550e-85,0.193958,3.679684e-85,True
5,serious_complaints,0.033036,0.015051,8.461224e-08,8.931149e-21,0.067558,1.091585e-20,True
7,cancel_intent,0.766903,0.180468,0.000000e+00,0.000000e+00,0.567343,0.000000e+00,True
8,price_discussions,0.304094,0.262497,2.809275e-06,2.679680e-16,0.051053,2.947648e-16,True
9,discount_requests,0.122089,0.090476,3.598342e-08,4.612966e-22,0.062271,6.342828e-22,True
10,recent_call_ratio,0.019335,0.010059,1.021882e-31,3.841942e-77,0.141111,7.043560e-77,True


**Cell 8: Display Significant Renewal Call Features**  
This cell filters the numerical results to show only the features that remain significant after p-value correction.  
- **Purpose**: Highlight the renewal call features most strongly associated with churn.  
- **Expected output**: a small summary table of significant features only.

In [19]:
num_results.to_csv(
    "../../reports/renewal_calls_hypothesis_results.csv",
    index=False
)

cat_results.to_csv(
    "../../reports/renewal_calls_categorical_hypothesis_results.csv",
    index=False
)